# Diagnóstico Automatizado: IA no Estetoscópio Digital - Parte 1

### Instalação do Modelo spaCy para Português

In [ ]:
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 40.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Parte 1 – Frases de Sintomas e Extração de Informações

## Extração De Sintomas e Sugestão De Diagnósticos

Objetivo:

1. Criar um arquivo .txt com 10 relatos-base de pacientes (acrescidos de 4 casos complementares de validação de negação).
2. Criar uma ontologia/mapa de conhecimento em .csv.
3. Ler os relatos do arquivo .txt.
4. Identificar expressões relacionadas a sintomas.
5. Comparar os sintomas encontrados com a ontologia.
6. Sugerir uma possível condição com base na associação mais forte encontrada.
7. Tratar negação explícita ("não sinto", "nunca senti", etc.) via análise de dependência sintática do spaCy.

---

Decisões Técnicas e Conceitos Utilizados

O código foi desenvolvido para simular um sistema básico de apoio à identificação de sintomas, utilizando uma ontologia (mapa de conhecimento) como base para as associações entre expressões e possíveis condições. Os relatos são armazenados em .txt e a ontologia em .csv, permitindo separar os dados da lógica de processamento.

Foi utilizado Pandas para estruturar e manipular os dados da ontologia e spaCy para processamento de linguagem natural, principalmente a lematização, reduzindo variações das palavras e tornando a comparação mais consistente. Também foi aplicada uma etapa de normalização textual, removendo diferenças de maiúsculas, acentuação e espaços.

A lógica de decisão utiliza pontuação por associação: uma correspondência completa entre os dois elementos da ontologia recebe maior peso que uma correspondência parcial. Essa abordagem foi escolhida para evitar que um único sintoma isolado gere automaticamente múltiplas condições. A ontologia construída contém 15 associações entre pares de sintomas e doenças, cobrindo quatro condições cardíacas: *Infarto*, *Insuficiência Cardíaca*, *Angina* e *Arritmia Cardíaca*.

Tratamento de negação via análise de dependência sintática

Além do matching textual/lematizado, o sistema incorpora detecção de negação utilizando a árvore de dependências do spaCy. Tokens cujo lema pertence ao léxico de negação (`não`, `nunca`, `jamais`, `nem`, `tampouco`) definem o "cue" de negação. O escopo é coletado recursivamente a partir do verbo regente, interrompendo-se em verbos coordenados para não propagar a negação a orações independentes.

Quando todas as ocorrências de um sintoma da ontologia caem dentro desse escopo, o sintoma é marcado como `(NEGADO)` e **não pontua** para o diagnóstico. Isso evita falsos positivos como classificar "não sinto dor no peito" como sugestivo de Infarto.

Para validar essa lógica, foram adicionados 4 relatos complementares com negação explícita, além dos 10 relatos-base. A saída do sistema destaca visualmente a separação entre os dois blocos.

O resultado final é estruturado em um DataFrame e exportado para .csv, facilitando a análise e validação dos resultados.

> Nota: A célula seguinte demonstra, de forma didática, a árvore de dependências gerada pelo spaCy para uma frase com negação.

---


######Observação:
<small> Este projeto é uma simulação acadêmica. Os resultados NÃO representam diagnóstico médico real.

In [ ]:
# ============================================================
# 1. IMPORTAÇÃO E CONFIGURAÇÃO
# ============================================================
import os, re, unicodedata
import pandas as pd
import spacy

NOME_ARQUIVO_TXT = "sintomas_pacientes.txt"
NOME_ARQUIVO_CSV = "ontologia_sintomas.csv"
NOME_ARQUIVO_RESULTADOS = "resultados_diagnostico_parte1.csv"

# ============================================================
# 2. RELATOS DOS PACIENTES
# ============================================================
# --- 10 relatos-base (escopo oficial da Parte 1) ---
frases_pacientes = [
    "Há dois dias estou com uma dor no peito intensa que piora quando faço esforço físico.",
    "Sinto cansaço constante há uma semana, mesmo depois de descansar o dia todo.",
    "Estou com uma falta de ar muito forte e dificuldade para respirar desde ontem à noite.",
    "Sinto um aperto no tórax seguido de tontura quando subo as escadas da minha casa.",
    "Relato fadiga extrema e fraqueza no corpo nos últimos três dias.",
    "Estou sentindo palpitações no coração e uma dor no peito leve que vai e volta.",
    "Apresento uma falta de ar progressiva ao caminhar curtas distâncias no bairro.",
    "Sinto cansaço constante e falta de ar quando tento dormir à noite.",
    "Tenho um aperto no tórax constante acompanhado de suor frio desde hoje cedo.",
    "Relato uma dor no peito irradiando para o braço esquerdo há cerca de duas horas."
]

# --- Casos complementares: validação do tratamento de negação ---
# (teste de análise de dependência sintática do spaCy)
frases_negacao = [
    "Não sinto dor no peito, apenas um leve cansaço nas pernas.",
    "Não tenho falta de ar nem palpitações, somente um desconforto muscular.",
    "Nunca senti aperto no tórax, apenas um cansaço ocasional.",
    "Jamais senti dor no peito ou falta de ar, somente cansaço leve."
]

frases_pacientes.extend(frases_negacao)

N_RELATOS_BASE   = 10
N_CASOS_NEGACAO  = len(frases_negacao)

# Criação do arquivo TXT
with open(NOME_ARQUIVO_TXT, "w", encoding="utf-8") as arquivo_txt:
    for frase in frases_pacientes:
        arquivo_txt.write(frase + "\n")

print(f"[OK] Arquivo '{NOME_ARQUIVO_TXT}' criado/atualizado.")
print(f"[INFO] Relatos-base: {N_RELATOS_BASE} | Casos complementares (negação): {N_CASOS_NEGACAO} | Total: {len(frases_pacientes)}")

# ============================================================
# 3. ONTOLOGIA / MAPA DE CONHECIMENTO
# ============================================================
dados_ontologia = {
    "Sintoma_1": [
        "dor no peito", "dor no peito", "dor no peito", "dor no peito", "aperto no tórax",
        "aperto no tórax", "cansaço constante", "cansaço constante", "fadiga", "falta de ar",
        "falta de ar", "falta de ar", "palpitações", "palpitações", "aperto no tórax"
    ],
    "Sintoma_2": [
        "esforço físico", "braço esquerdo", "suor frio", "tontura", "suor frio", "tontura",
        "mesmo depois de descansar", "falta de ar", "fraqueza no corpo", "dificuldade para respirar",
        "caminhar", "esforço físico", "dor no peito", "tontura", "falta de ar"
    ],
    "Doenca_Associada": [
        "Infarto", "Infarto", "Infarto", "Infarto", "Infarto", "Infarto",
        "Insuficiência Cardíaca", "Insuficiência Cardíaca", "Insuficiência Cardíaca",
        "Angina", "Angina", "Angina", "Arritmia Cardíaca", "Arritmia Cardíaca", "Angina"
    ]
}

# Criação e limpeza da Ontologia
df_ontologia_inicial = pd.DataFrame(dados_ontologia)
df_ontologia_inicial = df_ontologia_inicial.drop_duplicates(subset=["Sintoma_1", "Sintoma_2", "Doenca_Associada"]).reset_index(drop=True)
df_ontologia_inicial.to_csv(NOME_ARQUIVO_CSV, index=False, encoding="utf-8")

print(f"[OK] Ontologia '{NOME_ARQUIVO_CSV}' criada/atualizada.")
print(f"[INFO] Quantidade de associações: {len(df_ontologia_inicial)}")

# ============================================================
# 4. CARREGAMENTO DO MODELO E DADOS
# ============================================================
try:
    nlp = spacy.load("pt_core_news_sm")
except OSError:
    print("\n[ERRO] O modelo 'pt_core_news_sm' não foi encontrado.")
    print("\nNo Google Colab, execute primeiro:")
    print("!pip install -q spacy")
    print("!python -m spacy download pt_core_news_sm")
    raise

print("[OK] Modelo spaCy em português carregado com sucesso.")

df_ontologia = pd.read_csv(NOME_ARQUIVO_CSV, encoding="utf-8")
print("[OK] Mapa de conhecimento carregado.")
print(f"[INFO] Registros da ontologia: {len(df_ontologia)}")

# Leitura do arquivo TXT
frases_lidas = []
with open(NOME_ARQUIVO_TXT, "r", encoding="utf-8") as arquivo_txt:
    for linha in arquivo_txt:
        linha_limpa = linha.strip()
        if linha_limpa:
            frases_lidas.append(linha_limpa)

print("[OK] Arquivo de relatos lido.")
print(f"[INFO] Relatos encontrados: {len(frases_lidas)} ({N_RELATOS_BASE} base + {len(frases_lidas) - N_RELATOS_BASE} de negação)")

# ============================================================
# 5. FUNÇÕES AUXILIARES E PREPARAÇÃO
# ============================================================
def normalizar_texto(texto):
    texto = str(texto).lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(caractere for caractere in texto if unicodedata.category(caractere) != "Mn")
    texto = re.sub(r"\s+", " ", texto)
    return texto

def lematizar_texto(texto, nlp_model):
    doc = nlp_model(texto)
    lemmas = [token.lemma_.lower() for token in doc if not token.is_punct]
    return " ".join(lemmas)

# --- Detecção de negação via análise de dependência sintática (Cap. 10) ---
CUES_NEGACAO = {"não", "nunca", "jamais", "nem", "tampouco"}

def _coletar_escopo_negacao(token, tokens_negados):
    """Coleta recursivamente os tokens sob escopo de um verbo/constituinte negado.
    Para em verbos coordenados (conj VERB/AUX) para não propagar a negação
    para orações independentes ligadas por 'mas', 'porém', etc."""
    tokens_negados.add(token.i)
    for filho in token.children:
        if filho.dep_ == "conj" and filho.pos_ in ("VERB", "AUX"):
            continue
        _coletar_escopo_negacao(filho, tokens_negados)

def detectar_tokens_negados(doc):
    """Retorna o conjunto de índices de tokens sob escopo de negação.
    Aceita qualquer papel sintático para o cue (advmod, neg, cc) — o que
    importa é o lema pertencer ao léxico de negação."""
    tokens_negados = set()
    for token in doc:
        if token.lemma_.lower() in CUES_NEGACAO:
            _coletar_escopo_negacao(token.head, tokens_negados)
    return tokens_negados

def expressao_esta_negada(expressao_normalizada, tokens_norm, tokens_relevantes, tokens_negados):
    """Verifica se TODAS as ocorrências da expressão no texto estão negadas.
    Compara TEXTO NORMALIZADO (não lemas), pois lemas de palavras funcionais
    (ex.: 'no' → 'em') quebrariam a correspondência com a ontologia."""
    if not tokens_negados:
        return False

    palavras = expressao_normalizada.split()
    n = len(palavras)
    if n == 0:
        return False

    achou_negada = False
    for i in range(len(tokens_norm) - n + 1):
        if tokens_norm[i:i + n] == palavras:
            span_idx = [tokens_relevantes[j].i for j in range(i, i + n)]
            if not any(idx in tokens_negados for idx in span_idx):
                return False  # existe pelo menos uma ocorrência NÃO-negada
            achou_negada = True
    return achou_negada

# Preparação das expressões da ontologia
df_ontologia["Sintoma_1_Normalizado"] = df_ontologia["Sintoma_1"].apply(normalizar_texto)
df_ontologia["Sintoma_2_Normalizado"] = df_ontologia["Sintoma_2"].apply(normalizar_texto)
df_ontologia["Sintoma_1_Lematizado"] = df_ontologia["Sintoma_1_Normalizado"].apply(lambda texto: lematizar_texto(texto, nlp))
df_ontologia["Sintoma_2_Lematizado"] = df_ontologia["Sintoma_2_Normalizado"].apply(lambda texto: lematizar_texto(texto, nlp))

def expressao_encontrada(expressao_normalizada, expressao_lematizada, texto_normalizado, texto_lematizado):
    if expressao_normalizada in texto_normalizado:
        return True
    if expressao_lematizada in texto_lematizado:
        return True
    return False

# ============================================================
# 6. FUNÇÃO PRINCIPAL DE PROCESSAMENTO
# ============================================================
def extrair_sintomas_e_diagnosticar(texto, ontologia_df, nlp_model):
    # Processa o texto ORIGINAL com spaCy (para análise de dependência)
    doc_original = nlp_model(texto)
    tokens_relevantes = [t for t in doc_original if not t.is_punct and not t.is_space]
    # Usa TEXTO NORMALIZADO (não lemas) para casar com a ontologia token-a-token
    tokens_norm = [normalizar_texto(t.text) for t in tokens_relevantes]

    # Detecta escopo de negação via dependência sintática
    tokens_negados = detectar_tokens_negados(doc_original)

    # Caminho de matching original (inalterado)
    texto_normalizado = normalizar_texto(texto)
    texto_lematizado = lematizar_texto(texto_normalizado, nlp_model)

    sintomas_encontrados = set()
    sintomas_negados = set()
    associacoes_completas = []
    pontuacoes_por_doenca = {}
    correspondencias_parciais = []

    for _, row in ontologia_df.iterrows():
        sintoma_1, sintoma_2, doenca = row["Sintoma_1"], row["Sintoma_2"], row["Doenca_Associada"]
        s1_normalizado, s2_normalizado = row["Sintoma_1_Normalizado"], row["Sintoma_2_Normalizado"]
        s1_lematizado, s2_lematizado = row["Sintoma_1_Lematizado"], row["Sintoma_2_Lematizado"]

        match_s1 = expressao_encontrada(s1_normalizado, s1_lematizado, texto_normalizado, texto_lematizado)
        match_s2 = expressao_encontrada(s2_normalizado, s2_lematizado, texto_normalizado, texto_lematizado)

        # Verificação de negação: só se aplica quando há match
        s1_negado = match_s1 and expressao_esta_negada(s1_normalizado, tokens_norm, tokens_relevantes, tokens_negados)
        s2_negado = match_s2 and expressao_esta_negada(s2_normalizado, tokens_norm, tokens_relevantes, tokens_negados)

        if s1_negado:
            sintomas_negados.add(sintoma_1)
        if s2_negado:
            sintomas_negados.add(sintoma_2)

        # Match efetivo = encontrado E não-negado
        match_s1_efetivo = match_s1 and not s1_negado
        match_s2_efetivo = match_s2 and not s2_negado

        if match_s1_efetivo and match_s2_efetivo:
            sintomas_encontrados.add(sintoma_1)
            sintomas_encontrados.add(sintoma_2)
            associacoes_completas.append(f"{sintoma_1} + {sintoma_2} → {doenca}")
            pontuacoes_por_doenca[doenca] = max(pontuacoes_por_doenca.get(doenca, 0), 3)

        elif match_s1_efetivo or match_s2_efetivo:
            if match_s1_efetivo:
                sintomas_encontrados.add(sintoma_1)
                correspondencias_parciais.append((sintoma_1, doenca))
            if match_s2_efetivo:
                sintomas_encontrados.add(sintoma_2)
                correspondencias_parciais.append((sintoma_2, doenca))

            if doenca not in pontuacoes_por_doenca:
                pontuacoes_por_doenca[doenca] = max(pontuacoes_por_doenca.get(doenca, 0), 1)

    # Definição do diagnóstico final
    if pontuacoes_por_doenca:
        maior_pontuacao = max(pontuacoes_por_doenca.values())
        diagnosticos_com_maior_pontuacao = [doenca for doenca, pontuacao in pontuacoes_por_doenca.items() if pontuacao == maior_pontuacao]
        diagnostico_final = " / ".join(sorted(diagnosticos_com_maior_pontuacao))
        pontuacao_final = maior_pontuacao
    else:
        diagnostico_final = "Diagnóstico Indeterminado"
        pontuacao_final = 0

    # Resultado da extração — sintomas com marcação de negação
    anotacoes = []
    for s in sorted(sintomas_encontrados):
        anotacoes.append(s)
    for s in sorted(sintomas_negados):
        anotacoes.append(f"{s} (NEGADO)")

    sintomas_str = ", ".join(anotacoes) if anotacoes else "Nenhum sintoma mapeado"
    associacoes_completas = list(dict.fromkeys(associacoes_completas))
    associacoes_str = " | ".join(associacoes_completas) if associacoes_completas else "Nenhuma associação completa encontrada"

    return (sintomas_str, diagnostico_final, pontuacao_final, associacoes_str)

# ============================================================
# 7. PROCESSAMENTO E EXIBIÇÃO DOS RESULTADOS
# ============================================================
resultados = []

for idx, frase in enumerate(frases_lidas, start=1):
    # --- Separador visual entre relatos-base e casos de validação de negação ---
    if idx == N_RELATOS_BASE + 1:
        print("\n\n")
        print("#" * 70)
        print("#" + " " * 68 + "#")
        print("#   CASOS COMPLEMENTARES — VALIDAÇÃO DO TRATAMENTO DE NEGAÇÃO        #")
        print("#   (Análise de dependência sintática do spaCy)                      #")
        print("#" + " " * 68 + "#")
        print("#" * 70)

    sintomas_extraidos, diagnostico, pontuacao, associacoes = extrair_sintomas_e_diagnosticar(frase, df_ontologia, nlp)

    resultados.append({
        "ID_Paciente": idx,
        "Relato_Paciente": frase,
        "Sintomas_Extraidos": sintomas_extraidos,
        "Diagnostico_Sugerido": diagnostico,
        "Pontuacao": pontuacao,
        "Associacoes_Encontradas": associacoes
    })

    print("\n" + "=" * 70)
    print(f"PACIENTE #{idx:02d}")
    print("=" * 70)
    print(f"Relato: {frase}")
    print(f"\nSintomas identificados: {sintomas_extraidos}")
    print(f"Diagnóstico sugerido: {diagnostico}")
    print(f"Pontuação: {pontuacao}")
    print(f"Associação encontrada: {associacoes}")

# ============================================================
# 8. SALVAMENTO E RESUMO FINAL
# ============================================================
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv(NOME_ARQUIVO_RESULTADOS, index=False, encoding="utf-8")

total_relatos = len(df_resultados)
relatos_com_sintomas = df_resultados[df_resultados["Sintomas_Extraidos"] != "Nenhum sintoma mapeado"].shape[0]
relatos_com_diagnostico = df_resultados[df_resultados["Diagnostico_Sugerido"] != "Diagnóstico Indeterminado"].shape[0]

print("\n")
print("=" * 70)
print("RESUMO DO PROCESSAMENTO")
print("=" * 70)
print(f"Total de relatos processados: {total_relatos}")
print(f"Relatos com sintomas identificados: {relatos_com_sintomas}")
print(f"Relatos com diagnóstico sugerido: {relatos_com_diagnostico}")
print(f"Arquivo de resultados: {NOME_ARQUIVO_RESULTADOS}")
print("\n[OK] Processamento da Parte 1 concluído com sucesso.")
print("\n[AVISO] Este resultado é uma simulação acadêmica baseada na ontologia criada para o projeto e não representa diagnóstico médico real.")

[OK] Arquivo 'sintomas_pacientes.txt' criado/atualizado.
[INFO] Relatos-base: 10 | Casos complementares (negação): 4 | Total: 14
[OK] Ontologia 'ontologia_sintomas.csv' criada/atualizada.
[INFO] Quantidade de associações: 15
[OK] Modelo spaCy em português carregado com sucesso.
[OK] Mapa de conhecimento carregado.
[INFO] Registros da ontologia: 15
[OK] Arquivo de relatos lido.
[INFO] Relatos encontrados: 14 (10 base + 4 de negação)

PACIENTE #01
Relato: Há dois dias estou com uma dor no peito intensa que piora quando faço esforço físico.

Sintomas identificados: dor no peito, esforço físico
Diagnóstico sugerido: Infarto
Pontuação: 3
Associação encontrada: dor no peito + esforço físico → Infarto

PACIENTE #02
Relato: Sinto cansaço constante há uma semana, mesmo depois de descansar o dia todo.

Sintomas identificados: cansaço constante, mesmo depois de descansar
Diagnóstico sugerido: Insuficiência Cardíaca
Pontuação: 3
Associação encontrada: cansaço constante + mesmo depois de descansar 

### Debug: inspeção da árvore de dependências

Célula de inspeção que demonstra como o spaCy estrutura uma frase com negação e quais tokens caem no escopo do cue `"não"`.

A propagação é conservadora: a negação alcança todo o escopo do verbo regente (`sinto`), sem interromper em marcadores como `"apenas"`. Isso é inofensivo porque, no pipeline principal, apenas sintomas presentes na ontologia são testados contra o escopo — mas vale registrar como direção para refinamento futuro.

In [ ]:
doc = nlp("Não sinto dor no peito, apenas um leve cansaço nas pernas.")
for t in doc:
    print(f"{t.text:12} | lemma={t.lemma_:10} | dep={t.dep_:10} | pos={t.pos_:8} | head={t.head.text}")
print("\nTokens negados:", detectar_tokens_negados(doc))

Não          | lemma=não        | dep=advmod     | pos=ADV      | head=sinto
sinto        | lemma=sinto      | dep=ROOT       | pos=VERB     | head=sinto
dor          | lemma=dor        | dep=obj        | pos=NOUN     | head=sinto
no           | lemma=em o       | dep=case       | pos=ADP      | head=peito
peito        | lemma=peito      | dep=nmod       | pos=NOUN     | head=dor
,            | lemma=,          | dep=punct      | pos=PUNCT    | head=cansaço
apenas       | lemma=apenas     | dep=advmod     | pos=ADV      | head=cansaço
um           | lemma=um         | dep=det        | pos=DET      | head=cansaço
leve         | lemma=leve       | dep=amod       | pos=ADJ      | head=cansaço
cansaço      | lemma=cansaço    | dep=obj        | pos=NOUN     | head=sinto
nas          | lemma=em o       | dep=case       | pos=ADP      | head=pernas
pernas       | lemma=perna      | dep=nmod       | pos=NOUN     | head=cansaço
.            | lemma=.          | dep=punct      | pos=PUNCT    | h